In [46]:
import pandas as pd
from operator import itemgetter
import networkx as nx

In [47]:
data = pd.read_csv('pre_survey.csv')

In [48]:
nodes_df = data[['ID','Name']]
nodes_df = nodes_df.rename(columns={'Name':'Label'})
print(len(nodes_df))

41


In [49]:
nodes_df.to_csv('nodes.csv', index=False)

In [50]:
working = data.iloc[:,4:]
working = working.iloc[:,working.columns!='Last modified time']
working = working.set_index('Name').drop(columns='NetID')

In [51]:
working.head()

,Nikhil Chinchalkar,Jason Wang,Rithya Sriram,Carina Lau,Jenny Williams,Rahi Dasgupta,Natalie Miller,Arnav Shah,Arjun Maitra,Aryan Shah,...,Eric Zhu,Ivy Liu,Tristan Albano,Winifred Agyei,Rhea Barot,Manya Pradeep Narayan,Adam Azevedo,Isha Nagireddy,Natan Kramskiy,Wonjin Eum
Name,,,,,,,,,,,,,,,,,,,,,
Nikhil Chinchalkar,I am this person,I speak with them at least once a week,I speak with them at least once a week,I speak with them at least once a week,I speak with them at least once a week,I speak with them at least once a week,I speak with them at least once a week,I've spoken to them more than once before,I've spoken to them more than once before,I've spoken to them more than once before,...,I recognize their face/name,I recognize their face/name,I recognize their face/name,I've spoken to them once before,I recognize their face/name,I recognize their face/name,I've spoken to them more than once before,I recognize their face/name,I've spoken to them once before,I've spoken to them once before
Tianyi Chen,I speak with them at least once a week,I've spoken to them more than once before,I've spoken to them more than once before,I've spoken to them more than once before,I've spoken to them more than once before,I've spoken to them more than once before,I've spoken to them once before,I've spoken to them more than once before,I've spoken to them more than once before,I've spoken to them more than once before,...,I've never seen/heard of this person before,I've never seen/heard of this person before,I've spoken to them once before,I've never seen/heard of this person before,I've never seen/heard of this person before,I've never seen/heard of this person before,I've never seen/heard of this person before,I've never seen/heard of this person before,I've never seen/heard of this person before,I've never seen/heard of this person before
Ethan Yang,I've never seen/heard of this person before,I've never seen/heard of this person before,I've never seen/heard of this person before,I've never seen/heard of this person before,I've never seen/heard of this person before,I've never seen/heard of this person before,I've never seen/heard of this person before,I've never seen/heard of this person before,I speak with them at least once a week,I recognize their face/name,...,I've never seen/heard of this person before,I've never seen/heard of this person before,I've never seen/heard of this person before,I've never seen/heard of this person before,I've never seen/heard of this person before,I've never seen/heard of this person before,I've never seen/heard of this person before,I've never seen/heard of this person before,I've never seen/heard of this person before,I've never seen/heard of this person before
Emily Fu,I've spoken to them once before,I've spoken to them once before,I've spoken to them more than once before,I've spoken to them more than once before,I've spoken to them more than once before,I've spoken to them more than once before,I've never seen/heard of this person before,I've never seen/heard of this person before,I recognize their face/name,I've never seen/heard of this person before,...,I've never seen/heard of this person before,I've never seen/heard of this person before,I've never seen/heard of this person before,I've never seen/heard of this person before,I've never seen/heard of this person before,I've never seen/heard of this person before,I've never seen/heard of this person before,I've never seen/heard of this person before,I've never seen/heard of this person before,I've never seen/heard of this person before
Tristan Albano,I've spoken to them once before,I've never seen/heard of this person before,I've never seen/heard of this person before,I've spoken to them once before,I've spoken to them once before,I've spoken to them once before,I've never seen/heard of this person before,I've never seen/heard of this person before,I've never seen/heard of this person before,I've never seen/heard of this person before,...,I've never see

In [52]:
test = working.stack().rename_axis(['Source','Target']).reset_index()


In [53]:
test.head()

,Source,Target,0
0,Nikhil Chinchalkar,Nikhil Chinchalkar,I am this person
1,Nikhil Chinchalkar,Jason Wang,I speak with them at least once a week
2,Nikhil Chinchalkar,Rithya Sriram,I speak with them at least once a week
3,Nikhil Chinchalkar,Carina Lau,I speak with them at least once a week
4,Nikhil Chinchalkar,Jenny Williams,I speak with them at least once a week


In [54]:
len(test['Source'].unique())

41

In [55]:
test = test.rename(columns={0:'Weight'})
test

,Source,Target,Weight
0,Nikhil Chinchalkar,Nikhil Chinchalkar,I am this person
1,Nikhil Chinchalkar,Jason Wang,I speak with them at least once a week
2,Nikhil Chinchalkar,Rithya Sriram,I speak with them at least once a week
3,Nikhil Chinchalkar,Carina Lau,I speak with them at least once a week
4,Nikhil Chinchalkar,Jenny Williams,I speak with them at least once a week
...,...,...,...
1881,Stella Ma,Manya Pradeep Narayan,I've never seen/heard of this person before
1882,Stella Ma,Adam Azevedo,I've never seen/heard of this person before
1883,Stella Ma,Isha Nagireddy,I've never seen/heard of this person before
1884,Stella Ma,Natan Kramskiy,I've never seen/heard of this person before


In [56]:
test['Weight'].unique()

array(['I am this person', 'I speak with them at least once a week',
       "I've spoken to them more than once before",
       'I recognize their face/name', "I've spoken to them once before",
       "I've never seen/heard of this person before",
       'I speak with them everyday'], dtype=object)

In [57]:
weights_map = {'I am this person':0,
               'I speak with them everyday':6,
               'I speak with them at least once a week':5,
               "I've spoken to them more than once before":4,
               "I've spoken to them once before":3,
               'I recognize their face/name':2,
               "I've never seen/heard of this person before":0}

In [58]:
test['Weight'] = test['Weight'].map(lambda x: weights_map[x])

In [59]:
test

,Source,Target,Weight
0,Nikhil Chinchalkar,Nikhil Chinchalkar,0
1,Nikhil Chinchalkar,Jason Wang,5
2,Nikhil Chinchalkar,Rithya Sriram,5
3,Nikhil Chinchalkar,Carina Lau,5
4,Nikhil Chinchalkar,Jenny Williams,5
...,...,...,...
1881,Stella Ma,Manya Pradeep Narayan,0
1882,Stella Ma,Adam Azevedo,0
1883,Stella Ma,Isha Nagireddy,0
1884,Stella Ma,Natan Kramskiy,0


In [60]:
test.to_csv('edges.csv', index=False)